# H&E ↔ PSR Registration — SIFT + pyvips

Direct feature-based registration pipeline. No heavy frameworks.

**Pipeline per pair:**
1. Read both images at a small thumbnail level (fast, low memory)
2. Detect SIFT features on both thumbnails
3. FLANN match + Lowe ratio test → RANSAC affine estimation
4. Scale the affine transform to full resolution
5. Apply to full-resolution PSR via pyvips (streaming — no memory blowup)
6. Save as pyramidal OME-TIFF

**Why this works when VALIS/wsireg did not:**
- No serial-sort architecture — works for any number of pairs including 1
- pyvips applies the transform tile-by-tile at full resolution — no OOM
- SIFT features are robust across H&E ↔ PSR stain differences

**Expected input — H&E and PSR in separate directories:**
```
HE_DIR/          PSR_DIR/
├── mouse01.tif  ├── mouse01.tif
└── mouse02.tif  └── mouse02.tif
```

## 1 — Configuration

In [ ]:
# ── Edit these before running ─────────────────────────────────────────────────

HE_DIR     = '/home/qasim/Desktop/Computer/Projects/Qasim/Ahmed/Virtual_Staining/ASBTi/QP_HE/export/wsi_rgb_test/testA'
PSR_DIR    = '/home/qasim/Desktop/Computer/Projects/Qasim/Ahmed/Virtual_Staining/ASBTi/QP_SR/export/wsi_rgb_test/testB'
OUTPUT_DIR = '/home/qasim/Desktop/Computer/Projects/Qasim/Ahmed/Virtual_Staining/ASBTi/reg_output'

# Set to '' if filenames have no stain keyword and stems match directly
HE_KEYWORD  = ''
PSR_KEYWORD = ''

# Pyramid level used for SIFT feature detection
# Higher level = smaller image = faster but fewer features
# For a 20k×20k image at 0.23 µm/px:
#   Level 3 → ~2.5k×2.5k  (recommended)
#   Level 4 → ~1.25k×1.25k (faster, fewer features)
REG_LEVEL = 3

# SIFT: max features to detect per image
# Increase if few matches are found on sparse tissue
SIFT_FEATURES = 10000

# RANSAC reprojection threshold in pixels (at REG_LEVEL resolution)
RANSAC_THRESH = 5.0

# Lowe ratio test threshold (lower = stricter matching)
LOWE_RATIO = 0.7

## 2 — Imports

In [ ]:
import re
from pathlib import Path

import cv2
import numpy as np
import tifffile
import pyvips
import matplotlib.pyplot as plt
from PIL import Image

## 3 — Discover and pair files

In [ ]:
def collect_tifs(directory):
    d = Path(directory)
    return sorted(d.glob('*.tif')) + sorted(d.glob('*.tiff'))


def strip_keyword(stem, keyword):
    if not keyword:
        return stem
    return re.sub(
        rf'[_\-.]?{re.escape(keyword)}[_\-.]?', '', stem, flags=re.IGNORECASE
    ).strip('_-.')


he_files  = collect_tifs(HE_DIR)
psr_files = collect_tifs(PSR_DIR)

he_map  = {strip_keyword(f.stem, HE_KEYWORD):  f for f in he_files}
psr_map = {strip_keyword(f.stem, PSR_KEYWORD): f for f in psr_files}

common_keys = sorted(set(he_map) & set(psr_map))
pairs = [(he_map[k], psr_map[k]) for k in common_keys]

print(f'H&E  directory : {HE_DIR}  ({len(he_files)} files)')
print(f'PSR  directory : {PSR_DIR}  ({len(psr_files)} files)')
print(f'Matched pairs  : {len(pairs)}\n')
for he, psr in pairs:
    print(f'  HE : {he.name}')
    print(f'  PSR: {psr.name}')
    print()

unmatched_he  = set(he_map)  - set(psr_map)
unmatched_psr = set(psr_map) - set(he_map)
if unmatched_he:
    print(f'Warning — unmatched H&E : {unmatched_he}')
if unmatched_psr:
    print(f'Warning — unmatched PSR : {unmatched_psr}')

## 4 — Registration functions

In [ ]:
def read_thumbnail(path, target_px=2500):
    '''
    Read a thumbnail with long side ~target_px using pyvips (streaming).
    Works for both flat TIFs and pyramidal TIFs — never loads full image.
    Returns (HWC uint8 RGB ndarray, downsample_factor).
    '''
    img = pyvips.Image.new_from_file(str(path), access='sequential')
    full_h, full_w = img.height, img.width
    scale = target_px / max(full_h, full_w)
    if scale < 1.0:
        img = img.resize(scale)
    arr = np.ndarray(
        buffer=img.write_to_memory(),
        dtype=np.uint8,
        shape=(img.height, img.width, img.bands),
    )
    downsample = max(full_h, full_w) / max(img.height, img.width)
    return arr[..., :3].copy(), downsample


def get_full_size(path):
    '''Return (height, width) without loading the image.'''
    img = pyvips.Image.new_from_file(str(path), access='sequential')
    return img.height, img.width


def sift_affine(he_img, psr_img, sift_features, lowe_ratio, ransac_thresh):
    '''
    Estimate affine transform mapping PSR → HE using SIFT + RANSAC.
    Returns (M, n_matches, n_inliers) where M is a 2×3 affine matrix.
    '''
    he_gray  = cv2.cvtColor(he_img,  cv2.COLOR_RGB2GRAY)
    psr_gray = cv2.cvtColor(psr_img, cv2.COLOR_RGB2GRAY)

    sift = cv2.SIFT_create(nfeatures=sift_features)
    kp_he,  des_he  = sift.detectAndCompute(he_gray,  None)
    kp_psr, des_psr = sift.detectAndCompute(psr_gray, None)

    if des_he is None or des_psr is None or len(des_he) < 4 or len(des_psr) < 4:
        raise RuntimeError('Too few SIFT features. Try increasing SIFT_FEATURES or THUMB_SIZE.')

    index_params  = dict(algorithm=1, trees=5)
    search_params = dict(checks=50)
    flann   = cv2.FlannBasedMatcher(index_params, search_params)
    matches = flann.knnMatch(des_psr, des_he, k=2)

    good = [m for m, n in matches if m.distance < lowe_ratio * n.distance]

    if len(good) < 6:
        raise RuntimeError(f'Only {len(good)} good matches. '
                           'Try increasing SIFT_FEATURES or loosening LOWE_RATIO.')

    src = np.float32([kp_psr[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([kp_he[m.trainIdx].pt  for m in good]).reshape(-1, 1, 2)

    M, mask = cv2.estimateAffine2D(src, dst, method=cv2.RANSAC,
                                   ransacReprojThreshold=ransac_thresh)

    if M is None:
        raise RuntimeError('RANSAC failed to find a valid affine transform.')

    return M, len(good), int(mask.sum())


def scale_affine(M, downsample):
    '''Scale 2×3 affine translation from thumbnail to full resolution.'''
    M_full = M.copy()
    M_full[0, 2] *= downsample
    M_full[1, 2] *= downsample
    return M_full


def apply_affine_pyvips(psr_path, M_full, he_h_full, he_w_full, dst_path):
    '''
    Apply forward affine (PSR → HE) to full-resolution PSR via pyvips streaming.
    pyvips uses the inverse transform internally (output → input coords).
    '''
    M_inv = cv2.invertAffineTransform(M_full)
    a, b, tx = M_inv[0]
    c, d, ty = M_inv[1]

    psr_vips = pyvips.Image.new_from_file(str(psr_path), access='sequential')

    warped = psr_vips.affine(
        [a, b, c, d],
        interpolate=pyvips.Interpolate.new('bilinear'),
        odx=tx,
        ody=ty,
        oarea=[0, 0, he_w_full, he_h_full],
    )

    warped.write_to_file(
        str(dst_path),
        compression='deflate',
        tile=True,
        pyramid=True,
        tile_width=512,
        tile_height=512,
    )


print('Registration functions loaded.')

## 5 — Register each pair

In [ ]:
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

registered_results = []   # list of (he_path, psr_path, reg_psr_path)

for i, (he, psr) in enumerate(pairs):
    pair_name = strip_keyword(he.stem, HE_KEYWORD)
    dst = out_dir / f'{pair_name}_PSR_registered.tiff'

    print(f'[{i+1}/{len(pairs)}] {pair_name}')

    if dst.exists():
        print(f'  [skip] already registered')
        registered_results.append((he, psr, dst))
        continue

    # Step 1 — read thumbnails
    print('  Reading thumbnails (pyvips streaming, target 2500 px) ...')
    he_thumb,  ds_he  = read_thumbnail(he,  target_px=2500)
    psr_thumb, ds_psr = read_thumbnail(psr, target_px=2500)
    print(f'  H&E  thumb: {he_thumb.shape}  (downsample ×{ds_he:.1f})')
    print(f'  PSR  thumb: {psr_thumb.shape}  (downsample ×{ds_psr:.1f})')

    # Step 2 — SIFT + RANSAC affine at thumbnail resolution
    print('  Detecting SIFT features and estimating affine ...')
    M_thumb, n_matches, n_inliers = sift_affine(
        he_thumb, psr_thumb, SIFT_FEATURES, LOWE_RATIO, RANSAC_THRESH
    )
    print(f'  Matches: {n_matches}  RANSAC inliers: {n_inliers}')

    # Step 3 — scale affine to full resolution
    M_full = scale_affine(M_thumb, ds_he)
    he_h_full, he_w_full = get_full_size(he)
    print(f'  Full-res H&E size: {he_w_full}×{he_h_full}')

    # Step 4 — apply to full-res PSR via pyvips
    print(f'  Warping and saving → {dst.name} ...')
    apply_affine_pyvips(psr, M_full, he_h_full, he_w_full, dst)

    registered_results.append((he, psr, dst))
    print(f'  Done.')

print('\nAll registrations complete.')

## 6 — QC overlays

In [ ]:
def checkerboard(a, b, n=8):
    h, w   = a.shape[:2]
    th, tw = h // n, w // n
    board  = a.copy()
    for r in range(n):
        for c in range(n):
            if (r + c) % 2 == 0:
                board[r*th:(r+1)*th, c*tw:(c+1)*tw] = b[r*th:(r+1)*th, c*tw:(c+1)*tw]
    return board


qc_dir = out_dir / 'qc'
qc_dir.mkdir(exist_ok=True)

for he, psr, reg_psr in registered_results:
    pair_name = strip_keyword(he.stem, HE_KEYWORD)

    he_arr,  _  = read_thumbnail(he,      target_px=1500)
    psr_arr, _  = read_thumbnail(psr,     target_px=1500)
    reg_arr, _  = read_thumbnail(reg_psr, target_px=1500)

    # Resize all to H&E thumbnail size
    h, w = he_arr.shape[:2]
    def rsz(arr):
        return np.array(Image.fromarray(arr).resize((w, h), Image.LANCZOS))
    psr_arr = rsz(psr_arr)
    reg_arr = rsz(reg_arr)
    board   = checkerboard(he_arr, reg_arr)

    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    fig.suptitle(pair_name, fontsize=13, fontweight='bold')
    for ax, img, title in zip(
        axes,
        [he_arr, psr_arr, reg_arr, board],
        ['H&E (fixed)', 'PSR (original)', 'PSR (registered)', 'Checkerboard H&E | PSR-reg'],
    ):
        ax.imshow(img)
        ax.set_title(title, fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    out_png = qc_dir / f'{pair_name}_qc.png'
    plt.savefig(str(out_png), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_png}')

## 7 — Summary

In [ ]:
print('=' * 60)
print(f'  Pairs processed : {len(registered_results)}')
print(f'  Output          : {out_dir}')
print(f'  QC overlays     : {qc_dir}')
print('=' * 60)
for he, psr, reg_psr in registered_results:
    print(f'  {strip_keyword(he.stem, HE_KEYWORD)} → {reg_psr.name}')